In [3]:
from pathlib import Path
import sys, shutil, subprocess, json
import pandas as pd
# ------------------------------------------------
# Paths
# ------------------------------------------------
REPO = Path.cwd().resolve()
while not (REPO / "src" / "brcm").is_dir():
    REPO = REPO.parent

MATLAB = shutil.which("matlab")
MATLAB = R"C:\Program Files\MATLAB\R2026a\bin\matlab.exe"

REPORT = REPO / "outputs/parity/simp/parity_report.json"



In [4]:
# ------------------------------------------------
# Run helpers
# ------------------------------------------------
rows = []

def run(name, cmd):
    r = subprocess.run(
        cmd,
        cwd=REPO,
        capture_output=True,
        text=True,
    )

    print(f"\n===== {name} STDOUT =====")
    print(r.stdout)

    print(f"\n===== {name} STDERR =====")
    print(r.stderr)

    return r

# ------------------------------------------------
# 1. MATLAB: run original BRCM on _simp.idf
# ------------------------------------------------
if MATLAB:
    run(
        "MATLAB reference",
        [
            MATLAB,
            "-batch",
            "addpath(fullfile(pwd,'origin_matlab','parity'));"
            "export_simp_reference();"
        ],
    )
else:
    rows.append({
        "Stage": "MATLAB reference",
        "Status": "WAITING",
        "Return code": None,
        "Message": "MATLAB not found on PATH",
    })

# ------------------------------------------------
# 2. Python: run Python BRCM on the same _simp.idf
# ------------------------------------------------
run(
    "Python reference",
    [
        sys.executable,
        "tools/parity/run_simp_python_reference.py",
    ],
)

# ------------------------------------------------
# 3. Compare MATLAB vs Python
# ------------------------------------------------
run(
    "Parity comparison",
    [
        sys.executable,
        "tools/parity/compare_simp_parity.py",
    ],
)

display(pd.DataFrame(rows))

# ------------------------------------------------
# 4. Clean parity-result DataFrame
# ------------------------------------------------
if REPORT.exists():
    report = json.loads(REPORT.read_text())

    comparison = pd.DataFrame([
        ["Same IDF", report.get("same_input_file")],
        ["Seven tables", report.get("tables_status")],
        ["State identifiers", report.get("state_identifiers_status")],
        ["Heat-flux identifiers", report.get("heat_flux_identifiers_status")],
        ["Boundaries", report.get("boundary_status")],
        ["A", report.get("A_status")],
        ["Bq", report.get("Bq_status")],
        ["Xcap", report.get("Xcap_status")],
        ["Simulation", report.get("simulation_status")],
        ["Overall", report.get("overall_status")],
    ], columns=["Comparison", "Result"])

    display(comparison)


===== MATLAB reference STDOUT =====
Attention: It is assumed here that every floor and ceiling are pairwise parallel.
Attention: It is assumed here that every Wall has a Tilt angle of 90 deg.
Exported _simp.idf MATLAB reference to C:\Users\s2589602\Downloads\2026\BRCMToolbox_adapted\outputs\parity\simp\matlab


===== MATLAB reference STDERR =====


===== Python reference STDOUT =====
Exported _simp.idf Python reference to C:\Users\s2589602\Downloads\2026\BRCMToolbox_adapted\outputs\parity\simp\python
SHA256: 2319b9c1033ad5cd5797e43a0a971d8ffa7f4410fe05fedd374eb36d7b668c37


===== Python reference STDERR =====


===== Parity comparison STDOUT =====
Overall FAIL
First mismatch: Table constructions: row 3, column 'thickness': MATLAB='0.1015', Python='0.1014984'


===== Parity comparison STDERR =====



""


,Comparison,Result
0,Same IDF,True
1,Seven tables,None
2,State identifiers,None
3,Heat-flux identifiers,None
4,Boundaries,None
5,A,None
6,Bq,None
7,Xcap,None
8,Simulation,None
9,Overall,None
